# ASG-WM — Colab A100 · **Runtime 2 of 2**

Tier-1 Ph-4&ndash;Ph-5 &rarr; Tier-2 &rarr; eval &rarr; figures & tables. Reads Runtime-1 checkpoints from Drive.


## 0 &middot; Setup


In [ ]:
# Mount Drive, point at the code, install deps, define a robust run() helper.
from google.colab import drive; drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/asgwm'        # upload your local code/ to {ROOT}/code
import sys; sys.path.insert(0, f'{ROOT}/code')
%cd {ROOT}/code
!pip -q install -r requirements.txt
!nvidia-smi -L
import subprocess
def run(cmd):
    print('$', cmd)
    if subprocess.run(cmd, shell=True).returncode:
        raise SystemExit('COMMAND FAILED: ' + cmd)
R = f'--override paths.root={ROOT}'
print('ROOT =', ROOT)

## 1 &middot; Config (same toggles as Runtime 1)


In [ ]:
# --- edit these ---
FIRST_RUN = True   # True = time-boxed (~fits two A100 sessions). False = full publication budgets.
N_EVENTS  = 800    # SEVIR training events. Start small; raise for the final run.
# Forbid a silent synthetic fallback: a real-SEVIR failure will RAISE, not waste the session.
REAL  = '--override data.dataset=sevir --override data.require_real=true'
T1BOX = '--override train.tier1.max_steps_per_phase=4000' if FIRST_RUN else ''
T2BOX = '--override train.tier2.max_steps=12000' if FIRST_RUN else ''
print('FIRST_RUN =', FIRST_RUN, '| N_EVENTS =', N_EVENTS)

## 2 &middot; Resume check &mdash; confirm Runtime-1 checkpoints exist


In [ ]:
import glob
for j in ['tier0_transition','tier1/ph3_asg','tier1/ph5_eqcot','tier2_endtoend']:
    fs = sorted(glob.glob(f'{ROOT}/ckpt/{j}/*.pt'))
    print(f'{j:22s}', fs[-1] if fs else '(none yet)')

## 3 &middot; Tier-1 Ph-4, Ph-5 (auto-seeds from the Ph-3 checkpoint)


In [ ]:
run(f"python scripts/20_train_tier1_curriculum.py {R} {T1BOX} --override train.tier1.phases='[\"ph4_cot\",\"ph5_eqcot\"]'")

## 4 &middot; Tier-2 (end-to-end + rectified-flow + intervention)
Resumable: if it doesn't finish, start a fresh A100 and re-run this cell. On a 16 GB T4 add `--override train.tier2.batch_size=4`.


In [ ]:
run(f'python scripts/30_train_tier2.py {R} {T2BOX}')

## 5 &middot; Evaluation (skill + faithfulness)


In [ ]:
run(f'python scripts/40_eval_skill.py {R}')
run(f'python scripts/41_eval_faithfulness.py {R}')

## 6 &middot; Figures + tables from real results


In [ ]:
run(f'python scripts/42_make_figures.py --gallery {R}')

## &check; Done &mdash; bring results home
`MyDrive/asgwm/results/figures/*.pdf` &rarr; local `draft/`; `results/tables/*.tex` &rarr; paste into Tables 2 & 4; recompile (TUTORIAL Part C).


In [ ]:
run(f'ls -1 {ROOT}/results/figures {ROOT}/results/tables')